# GA + Optuna Feeder Bus Route Optimization

This notebook separates Optuna hyperparameter tuning from final model execution.

## Normal workflow

1. For the first tuning run, set `RUN_OPTUNA_TUNING = True`.
2. After tuning, the notebook exports `*_selected_hyperparameters.csv`.
3. For later reruns, set `RUN_OPTUNA_TUNING = False`.
4. The final `Program_Running_Time_s` in `*_summary.csv` excludes Optuna tuning time.

The current default is `TMAX_MINUTES = 45.0` and `RUN_OPTUNA_TUNING = False`.


In [2]:

# ================================================================
# Common utilities for Feeder Bus Network Design Optimization
# Objective: Maximize Priority Index (PI)
# Constraint: closed route from/to START_STATION_ID with Tmax budget
# ================================================================

import os
import json
import math
import time
import random
import heapq
import shutil
import zipfile
import subprocess
import sys
import xml.sax.saxutils as saxutils
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    import folium
except Exception:
    folium = None

# -------------------------
# Editable experiment config
# -------------------------
INPUT_FILE_NAME = "INPUT DATA.csv"
INPUT_PATH = f"/content/{INPUT_FILE_NAME}"
LOCAL_FALLBACK_INPUT_PATH = "/mnt/data/INPUT DATA.csv"

START_STATION_ID = 31             # Metro Station National University
TMAX_MINUTES = 45               # Change this manually when needed
TMAX_SECONDS = TMAX_MINUTES * 60.0

R_EARTH_M = 6371000.0

# Search settings: change when you want a deeper or faster experiment
N_OPTUNA_TRIALS = 100              # Recommended thesis run: 50-100 if time allows
N_EVAL_SEEDS_PER_TRIAL = 2        # Average stochastic performance per Optuna trial
FINAL_RUN_SEEDS = [101]            # Final run using selected parameters. Use list(range(101, 111)) for robustness testing.
RANDOM_SEED = 42

# ------------------------------------------------------------
# Optuna separation mode
# ------------------------------------------------------------
# First-time tuning workflow:
#   1) Set RUN_OPTUNA_TUNING = True.
#   2) Run the notebook once to let Optuna find parameters.
#   3) The notebook exports GA_selected_hyperparameters.csv.
#   4) For later reruns, set RUN_OPTUNA_TUNING = False.
#
# Final comparison workflow:
#   RUN_OPTUNA_TUNING = False skips Optuna completely and uses the saved/manual
#   parameters below. The summary runtime then excludes all tuning time.
RUN_OPTUNA_TUNING = False
LOAD_SELECTED_PARAMS_FROM_CSV =True
SAVED_SELECTED_PARAMS_CSV = '/content/GA_selected_hyperparameters.csv'   # Empty = use /content/ga_optuna_feeder_bus_outputs/GA_selected_hyperparameters.csv
FALLBACK_TO_MANUAL_SELECTED_PARAMS = True

# Number of final independent runs using the selected parameters.
# Program_Running_Time_s in the final summary is the runtime of the best final run only.
# Final_Experiment_Runtime_s in the hyperparameter summary is the sum of all final seed runs.

# If True, backward arcs inherit the demand/PI/area metrics of their original forward source link.
COPY_BACKWARD_METRICS_FROM_FORWARD_SOURCE = True
ENFORCE_SIMPLE_NODE_ROUTE = True


def ensure_package(package_name: str):
    '''Install a missing package in Colab-like environments.'''
    try:
        __import__(package_name)
    except ImportError:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


def resolve_input_path():
    if os.path.exists(INPUT_PATH):
        return INPUT_PATH
    if os.path.exists(LOCAL_FALLBACK_INPUT_PATH):
        return LOCAL_FALLBACK_INPUT_PATH
    try:
        from google.colab import files
        print(f"Please upload {INPUT_FILE_NAME}")
        uploaded = files.upload()
        if INPUT_FILE_NAME in uploaded:
            return INPUT_PATH
    except Exception:
        pass
    raise FileNotFoundError(f"{INPUT_FILE_NAME} was not found. Upload it to Colab or place it under /content.")


def resolve_output_dir(folder_name: str):
    if os.path.exists('/content'):
        outdir = f'/content/{folder_name}'
    else:
        outdir = f'/mnt/data/{folder_name}'
    os.makedirs(outdir, exist_ok=True)
    return outdir


def haversine_m(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dl / 2) ** 2
    return 2 * R_EARTH_M * math.asin(math.sqrt(max(0, min(1, a))))


def load_and_prepare_input(path: str, start_id: int = START_STATION_ID):
    '''Read INPUT DATA.csv and calculate PD, PI, demand key, and area metrics.

    The PI construction follows the Hemdan-style ranking idea:
    1. Calculate potential demand PD_ij = Pop_ij / S_ij * L_ij.
    2. Rank unique non-zero demand links by PD_ij.
    3. Use the rank as PI_ij.

    Backward links are permitted as route arcs but share the same demand_key as their
    original forward source link. This prevents selecting both the forward and backward
    version of the same original link in one final route.
    '''
    raw = pd.read_csv(path)

    required_cols = [
        'link_name', 'link_type',
        'from_station_id', 'to_station_id',
        'from_lat', 'from_lon', 'to_lat', 'to_lon',
        'from_station_name', 'to_station_name',
        'gis_route_distance_m', 'average_travel_time_s',
        'assigned_grid_area_km2', 'Popij', 'Sij_m', 'forward_source_link_name'
    ]
    missing = [c for c in required_cols if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns in input CSV: {missing}")

    df = raw.copy()
    df['link_type'] = df['link_type'].astype(str).str.strip()

    for c in ['from_station_id', 'to_station_id']:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype(int)

    numeric_cols = [
        'from_lat', 'from_lon', 'to_lat', 'to_lon',
        'gis_route_distance_m', 'average_travel_time_s',
        'assigned_grid_area_km2', 'Popij', 'Sij_m'
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)

    # Station master table for Google My Maps export.
    station_rows = []
    for _, r in df.iterrows():
        station_rows.append({
            'station_id': int(r.from_station_id),
            'lat': float(r.from_lat),
            'lon': float(r.from_lon),
            'station_name': str(r.from_station_name)
        })
        station_rows.append({
            'station_id': int(r.to_station_id),
            'lat': float(r.to_lat),
            'lon': float(r.to_lon),
            'station_name': str(r.to_station_name)
        })
    stations = pd.DataFrame(station_rows).drop_duplicates('station_id').sort_values('station_id').reset_index(drop=True)

    if start_id not in set(stations['station_id']):
        raise ValueError(f"START_STATION_ID={start_id} is not found in station data.")

    start_row = stations[stations.station_id.eq(start_id)].iloc[0]
    start_lat, start_lon = float(start_row.lat), float(start_row.lon)

    df['mid_lat'] = (df['from_lat'] + df['to_lat']) / 2
    df['mid_lon'] = (df['from_lon'] + df['to_lon']) / 2
    df['Lij_to_station_m'] = [
        haversine_m(mlat, mlon, start_lat, start_lon)
        for mlat, mlon in zip(df.mid_lat, df.mid_lon)
    ]

    is_uturn = df['link_type'].str.lower().eq('u-turn')
    valid_pd = (~is_uturn) & (df.Popij > 0) & (df.Sij_m > 0)
    df['PD_base'] = 0.0
    df.loc[valid_pd, 'PD_base'] = df.loc[valid_pd, 'Popij'] / df.loc[valid_pd, 'Sij_m'] * df.loc[valid_pd, 'Lij_to_station_m']

    # Demand keys enforce forward/backward exclusion.
    df['demand_key'] = df['link_name'].astype(str)
    df['metric_source_link_name'] = df['link_name'].astype(str)

    source_col = df['forward_source_link_name'].fillna('').astype(str).str.strip()
    is_backward = df['link_type'].str.lower().eq('backward') & source_col.ne('')
    if COPY_BACKWARD_METRICS_FROM_FORWARD_SOURCE:
        df.loc[is_backward, 'demand_key'] = source_col[is_backward]
        df.loc[is_backward, 'metric_source_link_name'] = source_col[is_backward]

    df.loc[is_uturn, 'demand_key'] = 'UTURN_' + df.loc[is_uturn, 'link_name'].astype(str)
    df.loc[is_uturn, 'metric_source_link_name'] = df.loc[is_uturn, 'link_name'].astype(str)

    # Copy metrics from the source forward link. If the source link is not found, use the current row.
    source_df = df.set_index('link_name')
    metric_rows = []
    for _, r in df.iterrows():
        if str(r.link_type).lower() == 'u-turn':
            metric_rows.append({'PDij': 0.0, 'Pop_key': 0.0, 'Area_key': 0.0})
            continue
        src_name = str(r.metric_source_link_name)
        if src_name in source_df.index:
            sr = source_df.loc[src_name]
            if isinstance(sr, pd.DataFrame):
                sr = sr.iloc[0]
        else:
            sr = r
        metric_rows.append({
            'PDij': float(sr.PD_base),
            'Pop_key': float(sr.Popij),
            'Area_key': float(sr.assigned_grid_area_km2)
        })
    df = pd.concat([df, pd.DataFrame(metric_rows)], axis=1)

    # Rank unique positive demand keys into PI values.
    key_metrics = df[~is_uturn].drop_duplicates('demand_key')[['demand_key', 'PDij']].copy()
    positive = key_metrics[key_metrics.PDij > 0].sort_values(['PDij', 'demand_key']).reset_index(drop=True)
    positive['PIij'] = np.arange(1, len(positive) + 1)
    df = df.merge(positive[['demand_key', 'PIij']], on='demand_key', how='left')
    df['PIij'] = df['PIij'].fillna(0).astype(int)
    df.loc[is_uturn, ['PDij', 'Pop_key', 'Area_key', 'PIij']] = 0

    df = df.reset_index(drop=True)
    df['edge_id'] = df.index.astype(int)
    return df, stations


class FeederGraph:
    def __init__(self, edges_df: pd.DataFrame, stations_df: pd.DataFrame, start_id: int = START_STATION_ID):
        self.edges = edges_df.copy()
        self.stations = stations_df.copy()
        self.start_id = int(start_id)
        self.adj = defaultdict(list)
        self.rev = defaultdict(list)
        self.edge_by_id = {}
        self.nodes = set()

        for _, r in self.edges.iterrows():
            edge = {
                'edge_id': int(r.edge_id),
                'link_name': str(r.link_name),
                'link_type': str(r.link_type),
                'u': int(r.from_station_id),
                'v': int(r.to_station_id),
                'time_s': float(r.average_travel_time_s),
                'dist_m': float(r.gis_route_distance_m),
                'PI': int(round(float(r.PIij))),
                'PD': float(r.PDij),
                'Pop': float(r.Pop_key),
                'Area': float(r.Area_key),
                'demand_key': str(r.demand_key),
                'from_lat': float(r.from_lat),
                'from_lon': float(r.from_lon),
                'to_lat': float(r.to_lat),
                'to_lon': float(r.to_lon),
                'from_station_name': str(r.from_station_name),
                'to_station_name': str(r.to_station_name),
            }
            self.edge_by_id[edge['edge_id']] = edge
            self.adj[edge['u']].append(edge)
            self.rev[edge['v']].append(edge)
            self.nodes.add(edge['u'])
            self.nodes.add(edge['v'])

        for u in self.adj:
            self.adj[u].sort(key=lambda e: ((e['PI'] + 1) / (e['time_s'] + 1), e['PI']), reverse=True)

        unique_keys = self.edges.drop_duplicates('demand_key')
        self.total_available_PI = int(unique_keys['PIij'].sum())
        self.total_available_PD = float(unique_keys['PDij'].sum())
        self.total_available_Pop = float(unique_keys['Pop_key'].sum())
        self.total_available_Area = float(unique_keys['Area_key'].sum())
        self.positive_key_count = int(self.edges[self.edges.PIij > 0].demand_key.nunique())
        self.lb_to_start = self.shortest_time_to_start()

    def shortest_time_to_start(self):
        dist = {n: math.inf for n in self.nodes}
        dist[self.start_id] = 0.0
        pq = [(0.0, self.start_id)]
        while pq:
            d, n = heapq.heappop(pq)
            if d > dist[n] + 1e-9:
                continue
            for e in self.rev.get(n, []):
                nd = d + e['time_s']
                if nd < dist[e['u']] - 1e-9:
                    dist[e['u']] = nd
                    heapq.heappush(pq, (nd, e['u']))
        return dist

    def route_nodes(self, route):
        if not route:
            return []
        return [route[0]['u']] + [e['v'] for e in route]

    def metrics(self, route):
        used_keys = set()
        total_pi = 0
        pd = 0.0
        pop = 0.0
        area = 0.0
        for e in route:
            k = e['demand_key']
            if e['PI'] > 0 and k not in used_keys:
                used_keys.add(k)
                total_pi += e['PI']
                pd += e['PD']
                pop += e['Pop']
                area += e['Area']
        travel_time_s = sum(e['time_s'] for e in route)
        total_dist_m = sum(e['dist_m'] for e in route)
        return {
            'Total_route_travel_time_s': travel_time_s,
            'Total_route_travel_time_min': travel_time_s / 60.0,
            'Total_route_distance_m': total_dist_m,
            'Total_PI': int(total_pi),
            'Total_Potential_Demand': float(pd),
            'Total_Served_Population': float(pop),
            'SAp_km2': float(area),
            'SAp_percent': float(area / self.total_available_Area * 100) if self.total_available_Area > 0 else 0.0,
            'Num_links': len(route),
            'Num_nodes_including_return': len(self.route_nodes(route)),
            'Route_node_sequence': ' -> '.join(map(str, self.route_nodes(route))),
            'Route_link_sequence': ' | '.join(e['link_name'] for e in route),
        }

    def is_feasible(self, route, tmax_seconds: float = None, enforce_simple: bool = ENFORCE_SIMPLE_NODE_ROUTE):
        if tmax_seconds is None:
            tmax_seconds = TMAX_SECONDS
        if not route:
            return False, 'empty_route'
        if route[0]['u'] != self.start_id or route[-1]['v'] != self.start_id:
            return False, 'not_closed_from_start'

        elapsed = 0.0
        used_edges = set()
        used_keys = set()
        visited_nodes = {self.start_id}
        previous = self.start_id

        for idx, e in enumerate(route):
            if e['u'] != previous:
                return False, 'not_continuous'
            previous = e['v']
            elapsed += e['time_s']
            if elapsed > tmax_seconds + 1e-6:
                return False, 'time_exceeded'
            if e['edge_id'] in used_edges:
                return False, 'duplicate_edge'
            used_edges.add(e['edge_id'])
            if e['PI'] > 0:
                if e['demand_key'] in used_keys:
                    return False, 'duplicate_forward_backward_key'
                used_keys.add(e['demand_key'])
            if enforce_simple:
                if e['v'] != self.start_id:
                    if e['v'] in visited_nodes:
                        return False, 'repeated_node'
                    visited_nodes.add(e['v'])
                elif idx != len(route) - 1:
                    return False, 'returned_start_before_final_edge'
        return True, 'OK'


def shortest_repair_path(graph: FeederGraph, source, remaining_time, visited_nodes, used_keys, used_edge_ids):
    '''Shortest-time feasible closure from current node back to the start station.

    Implementation note:
    heapq compares tuple elements from left to right. If two labels have the same
    cumulative time and node id, Python would otherwise try to compare the path
    object, which is a list of edge dictionaries, and raise:
    TypeError: '<' not supported between instances of 'dict' and 'dict'.

    A monotonic counter is therefore inserted as a deterministic tie-breaker.
    '''
    start = graph.start_id
    counter = 0
    pq = [(0.0, counter, source, [])]
    expansions = 0
    while pq and expansions < 8000:
        t, _, u, path = heapq.heappop(pq)
        expansions += 1
        if t > remaining_time + 1e-9:
            continue
        if u == start and path:
            return path
        path_nodes = {pe['v'] for pe in path if pe['v'] != start}
        path_keys = {pe['demand_key'] for pe in path if pe['PI'] > 0}
        path_edges = {pe['edge_id'] for pe in path}
        for e in graph.adj.get(u, []):
            if e['edge_id'] in used_edge_ids or e['edge_id'] in path_edges:
                continue
            if e['PI'] > 0 and (e['demand_key'] in used_keys or e['demand_key'] in path_keys):
                continue
            v = e['v']
            if v != start and (v in visited_nodes or v in path_nodes):
                continue
            nt = t + e['time_s']
            if nt > remaining_time + 1e-9:
                continue
            if v != start and nt + graph.lb_to_start.get(v, math.inf) > remaining_time + 1e-9:
                continue
            counter += 1
            heapq.heappush(pq, (nt, counter, v, path + [e]))
    return None


def randomized_construct(graph: FeederGraph, rng: random.Random, tmax_seconds: float = None,
                         alpha: float = 1.4, beta: float = 0.9, max_steps: int = 30,
                         close_prob: float = 0.05, temperature: float = 1.0):
    '''Construct a feasible closed route using PI/time-weighted randomized choices.'''
    if tmax_seconds is None:
        tmax_seconds = TMAX_SECONDS
    start = graph.start_id
    route = []
    visited = {start}
    used_keys = set()
    used_edges = set()
    current = start
    elapsed = 0.0

    for step in range(max_steps):
        remaining = tmax_seconds - elapsed
        repair = shortest_repair_path(graph, current, remaining, visited, used_keys, used_edges) if current != start else None

        candidates = []
        for e in graph.adj.get(current, []):
            if e['edge_id'] in used_edges:
                continue
            if e['PI'] > 0 and e['demand_key'] in used_keys:
                continue
            v = e['v']
            if v == start:
                if route and elapsed + e['time_s'] <= tmax_seconds + 1e-9:
                    candidates.append(e)
            else:
                if v in visited:
                    continue
                if elapsed + e['time_s'] + graph.lb_to_start.get(v, math.inf) <= tmax_seconds + 1e-9:
                    candidates.append(e)

        if not candidates:
            if repair:
                route.extend(repair)
                return route
            return None

        if repair and route and rng.random() < close_prob * (elapsed / max(tmax_seconds, 1e-9)):
            route.extend(repair)
            return route

        weights = []
        for e in candidates:
            score = ((e['PI'] + 1.0) ** alpha) / ((e['time_s'] + 1.0) ** beta)
            score *= rng.random() ** (1.0 / max(temperature, 1e-6))
            weights.append(max(score, 1e-12))
        selected = rng.choices(candidates, weights=weights, k=1)[0]

        route.append(selected)
        elapsed += selected['time_s']
        used_edges.add(selected['edge_id'])
        if selected['PI'] > 0:
            used_keys.add(selected['demand_key'])
        current = selected['v']
        if current == start:
            return route
        visited.add(current)

    if current != start:
        repair = shortest_repair_path(graph, current, tmax_seconds - elapsed, visited, used_keys, used_edges)
        if repair:
            route.extend(repair)
            return route
    return route if route and route[-1]['v'] == start else None


def fitness_value(graph: FeederGraph, route, tmax_seconds: float = None):
    '''Lexicographic scalar score: PI dominates; PD, area, and time are tie-breakers.'''
    if tmax_seconds is None:
        tmax_seconds = TMAX_SECONDS
    if route is None:
        return -1e18
    ok, _ = graph.is_feasible(route, tmax_seconds=tmax_seconds)
    if not ok:
        return -1e18
    m = graph.metrics(route)
    return (
        m['Total_PI'] * 1_000_000.0
        + m['Total_Potential_Demand'] * 1e-2
        + m['SAp_km2'] * 1e-1
        - m['Total_route_travel_time_s'] * 1e-5
    )


def route_links_df(graph: FeederGraph, route, model_name: str):
    rows = []
    cumulative_time = 0.0
    for sequence, e in enumerate(route, 1):
        cumulative_time += e['time_s']
        rows.append({
            'Model': model_name,
            'sequence': sequence,
            'link_name': e['link_name'],
            'link_type': e['link_type'],
            'from_station_id': e['u'],
            'to_station_id': e['v'],
            'from_station_name': e['from_station_name'],
            'to_station_name': e['to_station_name'],
            'from_lat': e['from_lat'],
            'from_lon': e['from_lon'],
            'to_lat': e['to_lat'],
            'to_lon': e['to_lon'],
            'travel_time_s': e['time_s'],
            'travel_time_min': e['time_s'] / 60.0,
            'cumulative_time_min': cumulative_time / 60.0,
            'distance_m': e['dist_m'],
            'PI': e['PI'],
            'Potential_Demand': e['PD'],
            'Served_Population': e['Pop'],
            'Served_Area_km2': e['Area'],
            'demand_key': e['demand_key'],
        })
    return pd.DataFrame(rows)


def route_nodes_df(graph: FeederGraph, route, model_name: str):
    station_dict = graph.stations.set_index('station_id').to_dict('index')
    rows = []
    for sequence, node_id in enumerate(graph.route_nodes(route), 1):
        s = station_dict.get(node_id, {})
        rows.append({
            'Model': model_name,
            'sequence': sequence,
            'station_id': node_id,
            'station_name': s.get('station_name', ''),
            'latitude': s.get('lat', np.nan),
            'longitude': s.get('lon', np.nan),
        })
    return pd.DataFrame(rows)


def route_coords_for_kml(route):
    coords = []
    if not route:
        return coords
    coords.append((route[0]['from_lon'], route[0]['from_lat'], 0))
    for e in route:
        coords.append((e['to_lon'], e['to_lat'], 0))
    return coords


def export_kml(graph: FeederGraph, route, path: str, model_name: str):
    coords_text = ' '.join(f"{lon},{lat},{alt}" for lon, lat, alt in route_coords_for_kml(route))
    metrics = graph.metrics(route)
    kml = f'''<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
<Document>
  <name>{saxutils.escape(model_name)} Feeder Bus Route</name>
  <Placemark>
    <name>{saxutils.escape(model_name)} Route | PI={metrics['Total_PI']} | Time={metrics['Total_route_travel_time_min']:.3f} min</name>
    <description>Total Potential Demand={metrics['Total_Potential_Demand']:.6f}; SAp={metrics['SAp_percent']:.6f}%</description>
    <LineString>
      <tessellate>1</tessellate>
      <coordinates>{coords_text}</coordinates>
    </LineString>
  </Placemark>
'''
    for _, row in route_nodes_df(graph, route, model_name).iterrows():
        kml += f'''
  <Placemark>
    <name>{int(row.sequence)}. Station {int(row.station_id)} - {saxutils.escape(str(row.station_name))}</name>
    <Point><coordinates>{row.longitude},{row.latitude},0</coordinates></Point>
  </Placemark>
'''
    kml += '</Document>\n</kml>\n'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(kml)


def export_geojson(graph: FeederGraph, route, path: str, model_name: str):
    features = []
    metrics = graph.metrics(route)
    features.append({
        'type': 'Feature',
        'properties': {'Model': model_name, **metrics},
        'geometry': {'type': 'LineString', 'coordinates': [[lon, lat] for lon, lat, _ in route_coords_for_kml(route)]}
    })
    for _, row in route_nodes_df(graph, route, model_name).iterrows():
        features.append({
            'type': 'Feature',
            'properties': row.to_dict(),
            'geometry': {'type': 'Point', 'coordinates': [float(row.longitude), float(row.latitude)]}
        })
    with open(path, 'w', encoding='utf-8') as f:
        json.dump({'type': 'FeatureCollection', 'features': features}, f, ensure_ascii=False, indent=2)


def export_html_map(graph: FeederGraph, route, path: str, model_name: str):
    if folium is None:
        print('folium is not available. Skipping HTML map export.')
        return
    start_row = graph.stations[graph.stations.station_id.eq(graph.start_id)].iloc[0]
    fmap = folium.Map(location=[start_row.lat, start_row.lon], zoom_start=14)
    m = graph.metrics(route)
    latlon = [(lat, lon) for lon, lat, _ in route_coords_for_kml(route)]
    folium.PolyLine(latlon, tooltip=f"{model_name}: PI={m['Total_PI']}, Time={m['Total_route_travel_time_min']:.2f} min").add_to(fmap)
    for _, row in route_nodes_df(graph, route, model_name).iterrows():
        folium.CircleMarker([row.latitude, row.longitude], radius=4, tooltip=f"#{int(row.sequence)} Station {int(row.station_id)}").add_to(fmap)
    fmap.save(path)


def export_all_outputs(graph: FeederGraph, route, outdir: str, model_name: str, runtime_s: float, hyperparams: dict, tuning_df: pd.DataFrame, extra_history_df: pd.DataFrame = None, final_experiment_runtime_s: float = None, optuna_tuning_runtime_s: float = 0.0):
    os.makedirs(outdir, exist_ok=True)
    metrics = graph.metrics(route)
    summary = {
        'Model': model_name,
        **metrics,
        'Program_Running_Time_s': runtime_s,
        'Runtime_Definition': 'Runtime of the selected best final run only; Optuna tuning runtime is excluded.',
        'Final_Experiment_Runtime_s': final_experiment_runtime_s if final_experiment_runtime_s is not None else runtime_s,
        'Optuna_Tuning_Runtime_s_excluded': optuna_tuning_runtime_s,
        'Tmax_minutes': TMAX_MINUTES,
        'Start_End_Station_ID': START_STATION_ID,
        'Selected_Hyperparameters_JSON': json.dumps(hyperparams),
        'Total_available_PI': graph.total_available_PI,
        'Total_available_Potential_Demand': graph.total_available_PD,
        'Total_available_assigned_grid_area_km2': graph.total_available_Area,
        'SAp_formula': 'sum(selected unique assigned_grid_area_km2) / sum(available unique assigned_grid_area_km2) * 100',
    }
    pd.DataFrame([summary]).to_csv(os.path.join(outdir, f'{model_name}_summary.csv'), index=False)
    route_links_df(graph, route, model_name).to_csv(os.path.join(outdir, f'{model_name}_route_links.csv'), index=False)
    route_nodes_df(graph, route, model_name).to_csv(os.path.join(outdir, f'{model_name}_route_nodes_for_google_mymaps.csv'), index=False)
    pd.DataFrame([hyperparams]).to_csv(os.path.join(outdir, f'{model_name}_selected_hyperparameters.csv'), index=False)
    tuning_df.to_csv(os.path.join(outdir, f'{model_name}_optuna_trials.csv'), index=False)
    if extra_history_df is not None and not extra_history_df.empty:
        extra_history_df.to_csv(os.path.join(outdir, f'{model_name}_best_convergence_history.csv'), index=False)
    export_kml(graph, route, os.path.join(outdir, f'{model_name}_route_map.kml'), model_name)
    export_geojson(graph, route, os.path.join(outdir, f'{model_name}_route_map.geojson'), model_name)
    export_html_map(graph, route, os.path.join(outdir, f'{model_name}_route_map.html'), model_name)

    metadata = {
        'input_file': INPUT_FILE_NAME,
        'rows': int(len(graph.edges)),
        'stations': int(len(graph.stations)),
        'start_end_station_id': START_STATION_ID,
        'Tmax_minutes': TMAX_MINUTES,
        'runtime_definition': 'Program_Running_Time_s excludes Optuna tuning and equals the selected best final run runtime.',
        'total_available_PI': graph.total_available_PI,
        'total_available_Potential_Demand': graph.total_available_PD,
        'total_available_served_population': graph.total_available_Pop,
        'total_available_assigned_grid_area_km2': graph.total_available_Area,
        'positive_demand_keys': graph.positive_key_count,
        'constraint_notes': [
            'closed route from/to START_STATION_ID',
            'total travel time <= Tmax',
            'no repeated directed edge',
            'no repeated positive demand_key, preventing forward/backward double counting',
            'simple node route except final return to START_STATION_ID'
        ],
    }
    with open(os.path.join(outdir, f'{model_name}_experiment_metadata.json'), 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    zip_path = os.path.join(outdir, f'{model_name}_outputs.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in os.listdir(outdir):
            full = os.path.join(outdir, fn)
            if os.path.isfile(full) and not fn.endswith('.zip'):
                zf.write(full, arcname=fn)

    print('\n=== Final Result ===')
    print(pd.DataFrame([summary])[[
        'Model', 'Total_route_travel_time_min', 'Total_PI', 'Total_Potential_Demand',
        'SAp_percent', 'Program_Running_Time_s', 'Runtime_Definition', 'Route_node_sequence'
    ]])
    print('\nOutputs saved to:', outdir)
    print('ZIP:', zip_path)
    return summary


# ================================================================
# Genetic Algorithm for Feeder Bus Network Design Problem
# Zhu (2017)-inspired circular route design with Optuna tuning
# ================================================================

# Colab installation cell equivalent for tuning mode only:
!pip -q install optuna folium

MODEL_NAME = 'GA'
OUTPUT_DIR_NAME = 'ga_optuna_feeder_bus_outputs'

# Manual selected parameters used when RUN_OPTUNA_TUNING = False and no CSV is found.
# Replace these values with your own Optuna-best parameters after the first tuning run.
GA_SELECTED_PARAMS = {
    'population_size': 140,
    'generations': 180,
    'elite_size': 14,
    'tournament_size': 4,
    'crossover_rate': 0.60,
    'mutation_rate': 0.70,
    'construct_alpha': 1.50,
    'construct_beta': 0.80,
    'close_prob': 0.05,
    'temperature': 1.10,
    'max_steps': 36,
}


def tournament_select(population, scores, rng, k=4):
    idxs = rng.sample(range(len(population)), min(k, len(population)))
    return population[max(idxs, key=lambda i: scores[i])]


def crossover_common_node(graph: FeederGraph, parent1, parent2, rng):
    """Route-preserving crossover using a common intermediate node."""
    n1 = graph.route_nodes(parent1)
    n2 = graph.route_nodes(parent2)
    common = set(n1[1:-1]).intersection(n2[1:-1])
    if not common:
        return parent1[:] if rng.random() < 0.5 else parent2[:]
    node = rng.choice(list(common))
    i1 = n1.index(node)
    i2 = n2.index(node)
    child = parent1[:i1] + parent2[i2:]
    if graph.is_feasible(child)[0]:
        return child
    child = parent2[:i2] + parent1[i1:]
    if graph.is_feasible(child)[0]:
        return child
    return parent1[:] if rng.random() < 0.5 else parent2[:]


def mutate_cut_regrow(graph: FeederGraph, route, rng, params):
    if rng.random() > params['mutation_rate'] or not route or len(route) <= 2:
        return route[:]
    cut = rng.randint(1, max(1, len(route) - 2))
    prefix = route[:cut]
    nodes = graph.route_nodes(prefix)
    current = prefix[-1]['v']
    if current == graph.start_id:
        return prefix
    elapsed = sum(e['time_s'] for e in prefix)
    visited = set(nodes)
    used_edges = {e['edge_id'] for e in prefix}
    used_keys = {e['demand_key'] for e in prefix if e['PI'] > 0}
    suffix = []

    for _ in range(max(1, params['max_steps'] - len(prefix))):
        remaining = TMAX_SECONDS - elapsed
        if current != graph.start_id and rng.random() < params['close_prob'] * (elapsed / max(TMAX_SECONDS, 1e-9)):
            repair = shortest_repair_path(graph, current, remaining, visited, used_keys, used_edges)
            if repair:
                suffix.extend(repair)
                break
        candidates = []
        for e in graph.adj.get(current, []):
            if e['edge_id'] in used_edges:
                continue
            if e['PI'] > 0 and e['demand_key'] in used_keys:
                continue
            v = e['v']
            if v == graph.start_id:
                if elapsed + e['time_s'] <= TMAX_SECONDS + 1e-9:
                    candidates.append(e)
            else:
                if v in visited:
                    continue
                if elapsed + e['time_s'] + graph.lb_to_start.get(v, math.inf) <= TMAX_SECONDS + 1e-9:
                    candidates.append(e)
        if not candidates:
            repair = shortest_repair_path(graph, current, TMAX_SECONDS - elapsed, visited, used_keys, used_edges)
            if repair:
                suffix.extend(repair)
            break
        weights = [((e['PI'] + 1.0) ** params['construct_alpha']) / ((e['time_s'] + 1.0) ** params['construct_beta']) for e in candidates]
        selected = rng.choices(candidates, weights=weights, k=1)[0]
        suffix.append(selected)
        elapsed += selected['time_s']
        used_edges.add(selected['edge_id'])
        if selected['PI'] > 0:
            used_keys.add(selected['demand_key'])
        current = selected['v']
        if current == graph.start_id:
            break
        visited.add(current)

    child = prefix + suffix
    return child if graph.is_feasible(child)[0] else route[:]


def run_ga_once(graph: FeederGraph, params: dict, seed: int):
    rng = random.Random(seed)
    t0 = time.perf_counter()
    population = []
    attempts = 0

    while len(population) < params['population_size'] and attempts < params['population_size'] * 100:
        attempts += 1
        r = randomized_construct(
            graph, rng,
            alpha=params['construct_alpha'],
            beta=params['construct_beta'],
            max_steps=params['max_steps'],
            close_prob=params['close_prob'],
            temperature=params['temperature']
        )
        if r and graph.is_feasible(r)[0]:
            population.append(r)

    if not population:
        return None, {'runtime_s': time.perf_counter() - t0, 'history': []}

    while len(population) < params['population_size']:
        population.append(rng.choice(population)[:])

    best = max(population, key=lambda r: fitness_value(graph, r))
    history = []

    for gen in range(params['generations']):
        scores = [fitness_value(graph, r) for r in population]
        gen_best = population[int(np.argmax(scores))]
        if fitness_value(graph, gen_best) > fitness_value(graph, best):
            best = gen_best[:]
        if gen % 10 == 0 or gen == params['generations'] - 1:
            bm = graph.metrics(best)
            history.append({'generation': gen, 'best_PI': bm['Total_PI'], 'best_time_min': bm['Total_route_travel_time_min']})

        elite_count = min(params['elite_size'], len(population))
        elite_idx = np.argsort(scores)[-elite_count:]
        new_population = [population[i][:] for i in elite_idx]

        while len(new_population) < params['population_size']:
            p1 = tournament_select(population, scores, rng, params['tournament_size'])
            p2 = tournament_select(population, scores, rng, params['tournament_size'])
            if rng.random() < params['crossover_rate']:
                child = crossover_common_node(graph, p1, p2, rng)
            else:
                child = p1[:]
            child = mutate_cut_regrow(graph, child, rng, params)
            if graph.is_feasible(child)[0]:
                new_population.append(child)
            else:
                fallback = randomized_construct(
                    graph, rng,
                    alpha=params['construct_alpha'],
                    beta=params['construct_beta'],
                    max_steps=params['max_steps'],
                    close_prob=params['close_prob'],
                    temperature=params['temperature']
                )
                new_population.append(fallback if fallback and graph.is_feasible(fallback)[0] else rng.choice(population)[:])
        population = new_population

    return best, {'runtime_s': time.perf_counter() - t0, 'history': history, 'params': params, 'seed': seed}


def suggest_ga_params(trial):
    population_size = trial.suggest_int('population_size', 60, 180, step=20)
    generations = trial.suggest_int('generations', 60, 180, step=20)
    elite_fraction = trial.suggest_float('elite_fraction', 0.06, 0.16)
    elite_size = max(2, int(round(population_size * elite_fraction)))
    return {
        'population_size': population_size,
        'generations': generations,
        'elite_size': elite_size,
        'tournament_size': trial.suggest_int('tournament_size', 2, 6),
        'crossover_rate': trial.suggest_float('crossover_rate', 0.35, 0.85),
        'mutation_rate': trial.suggest_float('mutation_rate', 0.35, 0.90),
        'construct_alpha': trial.suggest_float('construct_alpha', 0.80, 2.20),
        'construct_beta': trial.suggest_float('construct_beta', 0.30, 1.50),
        'close_prob': trial.suggest_float('close_prob', 0.01, 0.12),
        'temperature': trial.suggest_float('temperature', 0.70, 1.60),
        'max_steps': trial.suggest_int('max_steps', 18, 36),
    }


def objective_factory(graph: FeederGraph):
    def objective(trial):
        params = suggest_ga_params(trial)
        seed_scores = []
        seed_pis = []
        seed_times = []
        for offset in range(N_EVAL_SEEDS_PER_TRIAL):
            seed = RANDOM_SEED + trial.number * 100 + offset
            route, info = run_ga_once(graph, params, seed=seed)
            score = fitness_value(graph, route)
            seed_scores.append(score)
            if route:
                m = graph.metrics(route)
                seed_pis.append(m['Total_PI'])
                seed_times.append(m['Total_route_travel_time_min'])
            else:
                seed_pis.append(0)
                seed_times.append(np.nan)
        trial.set_user_attr('mean_PI', float(np.mean(seed_pis)))
        trial.set_user_attr('best_PI', int(np.max(seed_pis)))
        trial.set_user_attr('mean_time_min', float(np.nanmean(seed_times)))
        return float(np.mean(seed_scores))
    return objective


def suggest_ga_params_from_best_trial(best_trial):
    p = best_trial.params
    population_size = int(p['population_size'])
    elite_size = max(2, int(round(population_size * float(p.get('elite_fraction', 0.10)))))
    return {
        'population_size': population_size,
        'generations': int(p['generations']),
        'elite_size': elite_size,
        'tournament_size': int(p['tournament_size']),
        'crossover_rate': float(p['crossover_rate']),
        'mutation_rate': float(p['mutation_rate']),
        'construct_alpha': float(p['construct_alpha']),
        'construct_beta': float(p['construct_beta']),
        'close_prob': float(p['close_prob']),
        'temperature': float(p['temperature']),
        'max_steps': int(p['max_steps']),
    }


def normalize_ga_params(raw: dict):
    required = [
        'population_size', 'generations', 'elite_size', 'tournament_size', 'crossover_rate',
        'mutation_rate', 'construct_alpha', 'construct_beta', 'close_prob', 'temperature', 'max_steps'
    ]
    missing = [k for k in required if k not in raw or pd.isna(raw[k])]
    if missing:
        raise ValueError(f'Missing GA parameter(s): {missing}')
    return {
        'population_size': int(raw['population_size']),
        'generations': int(raw['generations']),
        'elite_size': int(raw['elite_size']),
        'tournament_size': int(raw['tournament_size']),
        'crossover_rate': float(raw['crossover_rate']),
        'mutation_rate': float(raw['mutation_rate']),
        'construct_alpha': float(raw['construct_alpha']),
        'construct_beta': float(raw['construct_beta']),
        'close_prob': float(raw['close_prob']),
        'temperature': float(raw['temperature']),
        'max_steps': int(raw['max_steps']),
    }


def load_selected_ga_params(outdir: str):
    candidate_path = SAVED_SELECTED_PARAMS_CSV.strip() or os.path.join(outdir, f'{MODEL_NAME}_selected_hyperparameters.csv')
    if LOAD_SELECTED_PARAMS_FROM_CSV and os.path.exists(candidate_path):
        row = pd.read_csv(candidate_path).iloc[0].to_dict()
        print(f'Loaded selected GA hyperparameters from CSV: {candidate_path}')
        return normalize_ga_params(row), f'CSV: {candidate_path}'
    if FALLBACK_TO_MANUAL_SELECTED_PARAMS:
        print('Using manual GA_SELECTED_PARAMS from the configuration section.')
        return normalize_ga_params(GA_SELECTED_PARAMS), 'Manual GA_SELECTED_PARAMS'
    raise FileNotFoundError(
        'No selected GA parameter CSV was found. Either set RUN_OPTUNA_TUNING=True once, '
        'upload GA_selected_hyperparameters.csv, or fill GA_SELECTED_PARAMS manually.'
    )


def run_ga_optuna_experiment():
    input_path = resolve_input_path()
    outdir = resolve_output_dir(OUTPUT_DIR_NAME)
    edges, stations = load_and_prepare_input(input_path, START_STATION_ID)
    graph = FeederGraph(edges, stations, START_STATION_ID)

    print('Feeder Bus Network Design Problem is treated as NP-hard; GA is used as a meta-heuristic search method.')
    print('Input rows:', len(edges))
    print('Stations:', len(stations))
    print('Total available PI:', graph.total_available_PI)
    print('Total available assigned area (km2):', round(graph.total_available_Area, 6))
    print('Tmax minutes:', TMAX_MINUTES)
    print('RUN_OPTUNA_TUNING:', RUN_OPTUNA_TUNING)

    optuna_tuning_runtime_s = 0.0
    best_optuna_value = np.nan
    best_optuna_trial_number = None
    selected_params_source = None
    trials_df = pd.DataFrame()
    study = None

    if RUN_OPTUNA_TUNING:
        ensure_package('optuna')
        import optuna
        from optuna.samplers import TPESampler
        tune_t0 = time.perf_counter()
        sampler = TPESampler(seed=RANDOM_SEED, multivariate=True, group=True)
        study = optuna.create_study(direction='maximize', sampler=sampler, study_name='GA_FeederBus_Maximize_PI')
        study.optimize(objective_factory(graph), n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
        optuna_tuning_runtime_s = time.perf_counter() - tune_t0
        selected_params = suggest_ga_params_from_best_trial(study.best_trial)
        best_optuna_value = study.best_value
        best_optuna_trial_number = study.best_trial.number
        selected_params_source = 'Optuna tuning in this run'
        trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
        trials_df.to_csv(os.path.join(outdir, f'{MODEL_NAME}_optuna_trials_raw.csv'), index=False)
    else:
        selected_params, selected_params_source = load_selected_ga_params(outdir)

    # Always save the selected parameters for future no-tuning reruns.
    pd.DataFrame([selected_params]).to_csv(os.path.join(outdir, f'{MODEL_NAME}_selected_hyperparameters.csv'), index=False)

    print('\nSelected GA hyperparameters source:', selected_params_source)
    print(json.dumps(selected_params, indent=2))

    # Final robustness run under selected parameters only.
    # This runtime excludes Optuna tuning by design.
    final_rows = []
    best_route = None
    best_info = None
    histories = []
    final_t0 = time.perf_counter()
    for seed in FINAL_RUN_SEEDS:
        route, info = run_ga_once(graph, selected_params, seed=seed)
        m = graph.metrics(route) if route else {}
        row = {'seed': seed, 'runtime_s': info['runtime_s'], **selected_params}
        row.update({k: m.get(k) for k in ['Total_PI', 'Total_route_travel_time_min', 'Total_Potential_Demand', 'Total_Served_Population', 'SAp_km2', 'SAp_percent', 'Route_node_sequence']})
        final_rows.append(row)
        if route and (best_route is None or fitness_value(graph, route) > fitness_value(graph, best_route)):
            best_route = route[:]
            best_info = info
        hist = pd.DataFrame(info.get('history', []))
        if not hist.empty:
            hist['seed'] = seed
            histories.append(hist)
    final_experiment_runtime_s = time.perf_counter() - final_t0

    if best_route is None:
        raise RuntimeError('No feasible GA route was found using the selected parameters. Increase max_steps/generations/population_size or rerun Optuna.')

    final_df = pd.DataFrame(final_rows)
    final_df.to_csv(os.path.join(outdir, f'{MODEL_NAME}_final_seed_runs.csv'), index=False)

    best_final_runtime_s = float(best_info.get('runtime_s', final_experiment_runtime_s)) if best_info else float(final_experiment_runtime_s)
    tuning_summary = pd.DataFrame([{
        'Model': MODEL_NAME,
        'Run_Mode': 'TUNE_AND_FINAL' if RUN_OPTUNA_TUNING else 'FINAL_ONLY_USING_SELECTED_PARAMS',
        'Selected_Params_Source': selected_params_source,
        'Optuna_n_trials': N_OPTUNA_TRIALS if RUN_OPTUNA_TUNING else 0,
        'Evaluation_seeds_per_trial': N_EVAL_SEEDS_PER_TRIAL if RUN_OPTUNA_TUNING else 0,
        'Optuna_Tuning_Runtime_s': optuna_tuning_runtime_s,
        'Final_run_seeds': len(FINAL_RUN_SEEDS),
        'Best_Optuna_Value': best_optuna_value,
        'Best_Optuna_Trial_Number': best_optuna_trial_number,
        **selected_params,
        'Final_Best_PI': int(final_df['Total_PI'].max()),
        'Final_Mean_PI': float(final_df['Total_PI'].mean()),
        'Final_Std_PI': float(final_df['Total_PI'].std(ddof=1)),
        'Best_Final_Run_Runtime_s': best_final_runtime_s,
        'Final_Mean_Runtime_s': float(final_df['runtime_s'].mean()),
        'Final_Experiment_Runtime_s': final_experiment_runtime_s,
        'Runtime_Note': 'Program_Running_Time_s in the model summary excludes Optuna tuning and equals Best_Final_Run_Runtime_s.'
    }])
    tuning_summary.to_csv(os.path.join(outdir, f'{MODEL_NAME}_hyperparameters_summary.csv'), index=False)

    all_history = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
    export_all_outputs(
        graph, best_route, outdir, MODEL_NAME,
        best_final_runtime_s, selected_params, trials_df, all_history,
        final_experiment_runtime_s=final_experiment_runtime_s,
        optuna_tuning_runtime_s=optuna_tuning_runtime_s,
    )
    return graph, best_route, selected_params, study, final_df


if __name__ == '__main__':
    run_ga_optuna_experiment()


Feeder Bus Network Design Problem is treated as NP-hard; GA is used as a meta-heuristic search method.
Input rows: 271
Stations: 92
Total available PI: 13366
Total available assigned area (km2): 12.97795
Tmax minutes: 45
RUN_OPTUNA_TUNING: False
Loaded selected GA hyperparameters from CSV: /content/GA_selected_hyperparameters.csv

Selected GA hyperparameters source: CSV: /content/GA_selected_hyperparameters.csv
{
  "population_size": 160,
  "generations": 180,
  "elite_size": 19,
  "tournament_size": 2,
  "crossover_rate": 0.432591268,
  "mutation_rate": 0.610768446,
  "construct_alpha": 0.831968822,
  "construct_beta": 0.804562617,
  "close_prob": 0.027575611,
  "temperature": 1.252181045,
  "max_steps": 35
}

=== Final Result ===
  Model  Total_route_travel_time_min  Total_PI  Total_Potential_Demand  \
0    GA                    44.488491      2815           432646.148166   

   SAp_percent  Program_Running_Time_s  \
0    22.878498               36.255566   

                        